# 10 · Fugas

**Objetivo (RF-002.2):** detectar las fugas de la red con el caudal que entrega la fuente y la
presión de los sensores, distinguirlas de las variaciones normales de la operación y decir en qué
tramo de la red están.

**Cómo se mide** (por episodio, como el detector del notebook 07):

| Métrica | Qué significa |
|---|---|
| fugas detectadas | episodios de fuga **visible** (algún sensor la nota) con alarma en alguna de esas horas |
| falsas alarmas por semana | episodios de alarma que no tocan ninguna hora con fuga visible |
| precisión, sensibilidad y F1 por evento | alarmas que eran fuga, fugas detectadas y su promedio (RNF-003, meta ≥ 80 %) |
| retraso | horas desde que la fuga se ve hasta la primera alarma |
| tramo correcto | el tramo dicho (entre qué dos sensores) es el de la fuga |

**Diseño** (`calidad_agua/fugas.py`):

1. **Regresión multivariable de la operación normal.** El caudal y la presión suben y bajan con la
   hora del día, el fin de semana, el calor y la lluvia, y con los arranques de bombas. Una
   regresión lineal por señal (caudal de la fuente y presión de cada sensor), ajustada con horas
   sin fuga del entrenamiento, da lo esperado; el **residuo** es lo que la operación normal no
   explica.
2. **Tres modelos con esos residuos**, como pide el RF-002.2: **Random Forest** y **SVM**
   (supervisados, con las etiquetas de fuga) y **Autoencoder** (TensorFlow/Keras, sin etiquetas:
   aprende a reconstruir la operación normal y marca lo que no puede reconstruir; umbral en el
   **percentil 95** del error en la operación normal).
3. **Umbral y horas de confirmación** con validación por bloques y la misma regla de los otros
   notebooks, fijada antes de ver resultados: máximo de fugas detectadas con a lo sumo 1 falsa
   alarma por semana en cada bloque.
4. **Tramo:** con 3 puntos de medición no se ubica una fuga al metro, pero sí entre qué dos
   sensores está (los dos más cercanos por tubería al punto de la fuga). Un Random Forest lo
   aprende del patrón de residuos; la línea base es "los dos sensores donde más bajó la presión".

Anticipar las fugas que empiezan pequeñas y crecen (24 h antes de que el tubo se rompa) se hace en
el notebook 09, con los demás tipos de evento.

In [ ]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))          # por si calidad_agua no está instalado (pip install -e .)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import joblib
from sklearn.metrics import mean_absolute_error, r2_score

from calidad_agua import carga, calidad, configuracion, detector, experimentos
from calidad_agua import fugas as F
from calidad_agua.eventos import episodios
from calidad_agua.graficos import estilo, guardar, COLORES, GRIS

estilo()
pd.set_option("display.precision", 2)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

crudo = carga.cargar_todo()
t15 = calidad.lecturas_operativas(crudo)
h = detector.horario(t15)                     # promedios horarios, eventos por sensor y partición
S = carga.sensores(h)
cat = carga.catalogo_eventos()
cat_f = cat[cat["tipo"] == "fuga"].copy()
y = F.clase(h)
PERIODO = {p: carga.periodos()[p] for p in carga.PARTICIONES}
entr = (h["particion"] == "entrenamiento").values
val = (h["particion"] == "validacion").values
prueba = (h["particion"] == "prueba").values

# horas "normales" para la regresión: sin fuga visible y lejos de una fuga (ni la fuga pequeña que
# crece antes, ni las 12 h después de repararla, mientras los tanques se recuperan)
cerca = np.zeros(len(h), bool)
for _, ev in cat_f.iterrows():
    cerca |= (h.index >= ev["inicio_precursor"]) & (h.index < ev["fin"] + pd.Timedelta(hours=12))
normal = ~cerca & (y.values == 0)

mods_normal = F.ajustar_normal(h, entr & normal)
X = F.tabla(h, mods_normal)
COL = list(X.columns)
TRAMO = F.tramos(carga.distancias_sensores())
cat_f["tramo"] = cat_f["ubicacion"].map(TRAMO)

vis = episodios(y)
vis["partición"] = h.loc[vis["inicio"], "particion"].values
print(f"Sensores del sistema: {S} · {len(COL)} variables")
pd.DataFrame({"fugas en el catálogo": cat_f.assign(p=h["particion"].reindex(cat_f["inicio"].dt.floor("h")).values)
              .groupby("p").size(), "episodios visibles": vis.groupby("partición").size()}) \
  .reindex(carga.PARTICIONES).fillna(0).astype(int)

In [ ]:
reg = experimentos.iniciar("fugas", "10 · fugas", "10_fugas")
reg.parametros({"sensores": ", ".join(S), "variables_entrada": len(COL)})

## 1. Lo normal y lo que deja una fuga

Qué tan bien explica la regresión el caudal y la presión en horas normales de la validación (que
no se usaron para ajustarla), y cómo se ve en promedio una fuga en los residuos: las horas antes y
después de que se empieza a ver.

In [ ]:
Xr, Y = F._regresores(h), F.senales(h)
filas = {}
for c in Y.columns:
    m = val & normal & Y[c].notna().values
    pred = mods_normal[c].predict(Xr.values[m])
    filas[c] = {"R² (validación, horas normales)": r2_score(Y[c].values[m], pred),
                "error medio absoluto": mean_absolute_error(Y[c].values[m], pred),
                "desviación del residuo": float(np.std(Y[c].values[m] - pred))}
calidad_regresion = pd.DataFrame(filas).T
reg.tabla(calidad_regresion.reset_index(), "regresion_normal")
calidad_regresion

In [ ]:
ventana = range(-48, 25)
dec = vis[vis["partición"] != "prueba"]
perfiles = {}
for c in ["caudal"] + [f"presion_{n}" for n in S]:
    serie = X[f"{c}_residuo_media3h"]
    perfiles[c] = pd.Series({k: np.nanmean([serie.get(ini + pd.Timedelta(hours=k), np.nan) for ini in dec["inicio"]])
                             for k in ventana})
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].plot(perfiles["caudal"], color=COLORES[0])
axes[0].set_ylabel("L/s"); axes[0].set_title("Caudal de la fuente: residuo (media 3 h)", fontsize=10)
for k, n in enumerate(S):
    axes[1].plot(perfiles[f"presion_{n}"], color=COLORES[k + 1], label=n)
axes[1].set_ylabel("m"); axes[1].set_title("Presión de cada sensor: residuo (media 3 h)", fontsize=10)
axes[1].legend(fontsize=7)
for a in axes:
    a.axvline(0, color=GRIS, ls="--", lw=1); a.axhline(0, color=GRIS, lw=0.8)
    a.set_xlabel("horas desde que la fuga se empieza a ver")
fig.suptitle(f"Promedio de {len(dec)} fugas visibles (entrenamiento y validación)", fontsize=10)
fig.tight_layout()
guardar(fig, "10_perfil_fuga"); plt.show()

## 2. Tres modelos y su umbral (validación por bloques)

Igual que en los notebooks 07 y 09: el entrenamiento se parte en cuatro bloques de 4 meses y, para
cada uno, se ajustan la regresión y los modelos con el resto y se evalúa en ese bloque; la
validación oficial es el quinto bloque. Con las probabilidades de los cinco bloques se elige, para
cada modelo, el umbral y las horas de confirmación con la regla fijada de antemano: **máximo de
fugas detectadas con a lo sumo 1 falsa alarma por semana en cada bloque**; a igualdad, menos
falsas y menos retraso. En el Autoencoder el umbral está en veces el percentil 95 del error normal
(1 = el percentil 95, el umbral que fija el RF-002.2).

La prueba no se usa para decidir.

In [ ]:
BLOQUES = {"sep–dic 2024": ("2024-09-01", "2025-01-01"), "ene–abr 2025": ("2025-01-01", "2025-05-01"),
           "may–ago 2025": ("2025-05-01", "2025-09-01"), "sep–dic 2025": ("2025-09-01", "2026-01-01"),
           "validación": tuple(str(f) for f in PERIODO["validacion"])}
ELECCION = F.HIPERPARAMETROS["eleccion"]
MODELOS = {"Random Forest": lambda X_, y_, n_: F.entrenar_bosque(X_, y_),
           "SVM": lambda X_, y_, n_: F.entrenar_svm(X_, y_),
           "Autoencoder": lambda X_, y_, n_: F.Autoencoder().fit(X_[n_])}
p_bloques = {m: pd.Series(np.nan, index=h.index) for m in MODELOS}
segundos = {m: 0.0 for m in MODELOS}
for nombre, (desde, hasta) in BLOQUES.items():
    en_bloque = (h.index >= desde) & (h.index < hasta)
    base = entr & ~en_bloque
    mods_b = F.ajustar_normal(h, base & normal)
    X_b = F.tabla(h, mods_b)
    for m, entrenar in MODELOS.items():
        t0 = time.time()
        mod_b = entrenar(X_b[base], y.values[base], (normal[base]))
        segundos[m] += time.time() - t0
        p_bloques[m][en_bloque] = F.probabilidad(mod_b, X_b[en_bloque])
    print(f"{nombre}: listo")
bloques = [(pd.Timestamp(a), pd.Timestamp(b)) for a, b in BLOQUES.values()]
en_bloques = np.zeros(len(h), bool)
for a, b in bloques:
    en_bloques |= (h.index >= a) & (h.index < b)

In [ ]:
eleccion, grillas, filas = {}, {}, []
for m in MODELOS:
    umbrales = ELECCION["umbrales_autoencoder"] if m == "Autoencoder" else ELECCION["umbrales"]
    p = p_bloques[m].fillna(0).values
    eleccion[m], grillas[m] = F.elegir_umbral(h.index, p, y.values, bloques, umbrales,
                                              ELECCION["confirmaciones_h"], ELECCION["max_falsas_por_semana"])
    u, c = eleccion[m]
    rs = [F.evaluar(h.index, F.alarma(p, u, c), y.values, a, b)[1] for a, b in bloques]
    det, n, fal, al = (sum(r[k] for r in rs) for k in ["detectadas", "fugas", "falsas", "alarmas"])
    semanas = en_bloques.sum() / (24 * 7)
    prec, sens = (al - fal) / al if al else np.nan, det / n
    filas.append({"modelo": m, "umbral": u, "confirmación (h)": c, "detectadas": det, "de": n,
                  "% detectadas": 100 * sens, "falsas por semana": fal / semanas,
                  "peor bloque (falsas/semana)": max(r["falsas_por_semana"] for r in rs),
                  "precisión %": 100 * prec, "F1 %": 100 * 2 * prec * sens / (prec + sens) if al else 0,
                  "entrenar (s, 5 bloques)": segundos[m]})
# el percentil 95 del Autoencoder sin ajustar (umbral 1, 1 h), como lo fija el RF-002.2
p_ae = p_bloques["Autoencoder"].fillna(0).values
rs = [F.evaluar(h.index, F.alarma(p_ae, 1.0, 1), y.values, a, b)[1] for a, b in bloques]
al, fal, det = sum(r["alarmas"] for r in rs), sum(r["falsas"] for r in rs), sum(r["detectadas"] for r in rs)
filas.append({"modelo": "Autoencoder, percentil 95 sin ajustar", "umbral": 1.0, "confirmación (h)": 1,
              "detectadas": det, "de": n, "% detectadas": 100 * det / n,
              "falsas por semana": fal / semanas, "peor bloque (falsas/semana)": max(r["falsas_por_semana"] for r in rs),
              "precisión %": 100 * (al - fal) / al if al else np.nan,
              "F1 %": 100 * 2 * ((al - fal) / al) * (det / n) / ((al - fal) / al + det / n) if al and det else 0,
              "entrenar (s, 5 bloques)": segundos["Autoencoder"]})
comparacion_bloques = pd.DataFrame(filas).set_index("modelo")
# el modelo elegido: el que detecta más dentro del límite; a igualdad, mayor F1
validos = comparacion_bloques.drop(index="Autoencoder, percentil 95 sin ajustar")
validos = validos[validos["peor bloque (falsas/semana)"] <= ELECCION["max_falsas_por_semana"]] \
    if (validos["peor bloque (falsas/semana)"] <= ELECCION["max_falsas_por_semana"]).any() else validos
ELEGIDO = validos.sort_values(["detectadas", "F1 %"], ascending=False).index[0]
UMBRAL, CONFIRMAR = eleccion[ELEGIDO]
print(f"Modelo elegido: {ELEGIDO}, umbral {UMBRAL:g}, confirmación {CONFIRMAR} h")
reg.parametros({"modelo": ELEGIDO, "umbral": UMBRAL, "confirmacion_h": CONFIRMAR}, "elegido")
reg.tabla(comparacion_bloques.reset_index(), "comparacion_bloques")
comparacion_bloques

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for k, m in enumerate(MODELOS):
    g = grillas[m]
    g = g.assign(semanas=en_bloques.sum() / (24 * 7))
    for c in ELECCION["confirmaciones_h"]:
        gg = g[g["confirmacion_h"] == c]
        ax.plot(gg["falsas"] / gg["semanas"], 100 * gg["detectadas"] / gg["fugas"], "o-", ms=3, lw=1,
                color=COLORES[k], alpha=1 if c == eleccion[m][1] else 0.35,
                label=f"{m}" if c == eleccion[m][1] else None)
ax.axvline(ELECCION["max_falsas_por_semana"], color=GRIS, ls="--", lw=1)
ax.set_xlabel("falsas alarmas por semana (cinco bloques)"); ax.set_ylabel("% de fugas detectadas")
ax.set_title("Fugas detectadas contra falsas alarmas, por umbral (confirmación elegida en color)", fontsize=10)
ax.legend(fontsize=7)
guardar(fig, "10_umbral"); plt.show()

## 3. Evaluación final en el periodo de prueba

Los tres modelos se entrenan con todo el entrenamiento, con su umbral y confirmación elegidos, y se
evalúan una sola vez en la prueba.

In [ ]:
finales, filas, eventos_prueba = {}, [], {}
desde, hasta = PERIODO["prueba"]
for m, entrenar in MODELOS.items():
    t0 = time.time()
    finales[m] = entrenar(X[entr], y.values[entr], normal[entr])
    p = F.probabilidad(finales[m], X)
    u, c = eleccion[m]
    ev, r = F.evaluar(h.index, F.alarma(p, u, c), y.values, desde, hasta)
    eventos_prueba[m] = ev.assign(p=[p[(h.index >= a) & (h.index <= b)].max() for a, b in zip(ev.inicio, ev.fin)])
    filas.append({"modelo": m, **r, "segundos": time.time() - t0})
    with reg.hija(f"10 · {m}") as hija:
        hija.parametros({"umbral": u, "confirmacion_h": c})
        hija.metricas({k: v for k, v in r.items() if np.isscalar(v) and not pd.isna(v)}, "prueba")
prueba_modelos = pd.DataFrame(filas).set_index("modelo")
for k in ["precision", "sensibilidad", "f1"]:
    prueba_modelos[k] = 100 * prueba_modelos[k]
reg.metricas({k: v for k, v in prueba_modelos.loc[ELEGIDO].items() if not pd.isna(v)}, "prueba")
reg.tabla(prueba_modelos.reset_index(), "prueba_modelos")
prueba_modelos.rename(columns={"precision": "precisión %", "sensibilidad": "sensibilidad %", "f1": "F1 %"})

In [ ]:
p_final = F.probabilidad(finales[ELEGIDO], X)
alarma_final = pd.Series(F.alarma(p_final, UMBRAL, CONFIRMAR), index=h.index)
tramo_p = slice(desde, hasta - pd.Timedelta(hours=1))
fig, axes = plt.subplots(2, 1, figsize=(11, 4.2), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
axes[0].plot(X.loc[tramo_p, "caudal_residuo_media3h"], color=COLORES[0], lw=0.8)
axes[0].set_ylabel("caudal: residuo\nmedia 3 h (L/s)")
for _, e in episodios(y.loc[tramo_p]).iterrows():
    for a in axes:
        a.axvspan(e.inicio, e.fin + pd.Timedelta(hours=1), color=COLORES[7], alpha=0.15, lw=0)
axes[1].fill_between(alarma_final.loc[tramo_p].index, 0, alarma_final.loc[tramo_p].astype(int), step="post",
                     color="#0b0b0b", lw=0)
axes[1].set_yticks([]); axes[1].set_ylabel("alarma")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
axes[0].set_title(f"Periodo de prueba: fugas visibles (franjas) y alarmas del {ELEGIDO}", fontsize=10)
fig.tight_layout()
guardar(fig, "10_linea_tiempo_prueba"); plt.show()

## 4. En qué tramo está la fuga

Cada fuga del catálogo se asigna al tramo entre los dos sensores más cercanos por tubería a su
punto (`calidad_agua/fugas.tramos`, con las distancias del simulador). Un Random Forest aprende el
tramo con las horas de fuga del entrenamiento; para cada fuga de la prueba se promedian sus
probabilidades en las horas en que la fuga se ve. La línea base es la regla "los dos sensores
donde más bajó la presión".

In [ ]:
# tramo de cada hora con fuga visible (la fuga del catálogo que está abierta en esa hora)
tramo_hora = pd.Series(np.nan, index=h.index, dtype=object)
for _, ev in cat_f.iterrows():
    m = (h.index >= ev["inicio"].floor("h")) & (h.index <= ev["fin"]) & (y.values == 1)
    tramo_hora[m] = ev["tramo"]
con_tramo = tramo_hora.notna().values
mod_tramo = F.entrenar_tramo(X[entr & con_tramo], tramo_hora[entr & con_tramo].values)
filas = []
for _, ev in cat_f[(cat_f["inicio"] >= desde) & (cat_f["inicio"] < hasta)].iterrows():
    horas = h.index[(h.index >= ev["inicio"].floor("h")) & (h.index <= ev["fin"]) & (y.values == 1)]
    if len(horas) == 0:
        continue                                      # ningún sensor la ve
    dicho, prob = F.tramo_por_evento(mod_tramo, X, horas)
    filas.append({"inicio": ev["inicio"], "nodo": ev["ubicacion"], "L/s": ev["intensidad"], "tramo real": ev["tramo"],
                  "tramo dicho": dicho, "probabilidad": prob, "regla": F.tramo_por_regla(X, S, horas)})
tramos_prueba = pd.DataFrame(filas)
acierto = (tramos_prueba["tramo real"] == tramos_prueba["tramo dicho"]).mean()
acierto_regla = (tramos_prueba["tramo real"] == tramos_prueba["regla"]).mean()
print(f"Tramo correcto: {100 * acierto:.0f} % con el modelo, {100 * acierto_regla:.0f} % con la regla "
      f"({len(tramos_prueba)} fugas visibles en la prueba)")
print("Tramos en el entrenamiento:", tramo_hora[entr].value_counts().to_dict())
reg.metricas({"acierto_tramo": acierto, "acierto_tramo_regla": acierto_regla}, "prueba")
reg.tabla(tramos_prueba, "tramos_prueba")
tramos_prueba

## 5. Modelo para el servicio

Se guardan la regresión de la operación normal, el modelo elegido con su umbral y confirmación y
el modelo de tramos. Si el elegido es el Autoencoder, la red se guarda aparte en formato Keras.

In [ ]:
carpeta = Path.cwd().parent / "modelos"
carpeta.mkdir(exist_ok=True)
modelo = finales[ELEGIDO]
paquete = {"nombre": ELEGIDO, "modelo": modelo, "columnas": COL, "normal": mods_normal, "umbral": UMBRAL,
           "confirmar_h": CONFIRMAR, "tramo": mod_tramo, "tramos": TRAMO.to_dict(), "sensores": S,
           "mlflow": reg.referencia()}
if isinstance(modelo, F.Autoencoder):
    modelo.red_.save(carpeta / "fugas_autoencoder.keras")
    red, modelo.red_ = modelo.red_, None
    joblib.dump(paquete, carpeta / "fugas.joblib", compress=3)
    modelo.red_ = red
    reg.archivo(carpeta / "fugas_autoencoder.keras")
else:
    joblib.dump(paquete, carpeta / "fugas.joblib", compress=3)
print(f"Guardado: modelos/fugas.joblib ({(carpeta / 'fugas.joblib').stat().st_size / 1e6:.1f} MB)")
reg.archivo(carpeta / "fugas.joblib")
reg.figuras_guardadas("10_")
reg.terminar()
print(f"Registrado en MLflow: experimento fugas, corrida {reg.id}")

## 6. Conclusiones

*(Se completan con los resultados de la corrida.)*